# Phase 2: closed-book and oracle LLM on PubMedQA dev

`Qwen/Qwen2.5-7B-Instruct` (pinned revision) in 4-bit NF4 on a T4, pure greedy decoding,
prompt v1. Settings are pre-registered in `docs/experiment-protocol.md` section 4 and
`configs/llm.yaml`.

* Runtime: **GPU, T4**. The first model download is about 15 GB.
* Every answer is cached on Google Drive. After a disconnect, re-run the same cells: finished
  questions come from the cache and the run continues where it stopped.
* The test split is locked: nothing here can load or score it.

## Step 1: settings (re-run after any session restart)

In [ ]:
import os

# Must be set before anything imports JAX (bm25s can pull it in later phases).
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

REPO_URL = "https://github.com/AyushYadav-rgb477/pubmedqa-rag-benchmark.git"
REF = "phase/2-llm"  # branch or tag to run
REPO_DIR = "/content/pubmedqa-rag-benchmark"
DRIVE_ROOT = "/content/drive/MyDrive/biomedqa"
RESULTS_DIR = f"{DRIVE_ROOT}/results/llm"
LLM_CACHE = f"{DRIVE_ROOT}/llm_cache"
HF_CACHE = "/content/hf_cache"

## Step 2: check the GPU and mount Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive

drive.mount("/content/drive")

## Step 3: get the code (public repo, no token)

In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin && git checkout -q {REF} && git pull -q --ff-only origin {REF}
!git log --oneline -1

## Step 4: install the pinned stack

In [ ]:
!pip uninstall -y -q torchvision torchaudio
!pip install -q "torch==2.6.0" "transformers==4.51.3" "accelerate==1.6.0" "bitsandbytes==0.45.5"
!pip install -q -e ".[llm,corpus,dev]"

## Step 5: check versions

If torch shows a different version: **Runtime > Restart session**, then Steps 1, 3 and 5.
If `cuda` is `False`: **Runtime > Change runtime type > T4 GPU**, then Steps 1 to 5 again.

In [ ]:
import accelerate
import bitsandbytes
import torch
import transformers

print(
    torch.__version__,
    transformers.__version__,
    accelerate.__version__,
    bitsandbytes.__version__,
    torch.cuda.is_available(),
)
assert torch.__version__.startswith("2.6.0")
assert transformers.__version__ == "4.51.3"
assert accelerate.__version__ == "1.6.0"
assert bitsandbytes.__version__ == "0.45.5"
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

## Step 6: tests on this stack, and verify the splits (must print `splits verified`)

In [ ]:
!python -m pytest -q
!python scripts/make_splits.py --cache-dir {HF_CACHE}

## Step 7: quick check with 3 questions (saves no results)

Downloads and loads the model. Check that `device_map` is `{'': 0}` and that the raw outputs
look like JSON answers.

In [ ]:
!python scripts/run_llm.py --system oracle --split dev --limit 3 --cache-dir {LLM_CACHE} --hf-cache-dir {HF_CACHE}

## Step 8: closed-book on dev (100 questions)

In [ ]:
!python scripts/run_llm.py --system closed_book --split dev --out {RESULTS_DIR} --cache-dir {LLM_CACHE} --hf-cache-dir {HF_CACHE}

## Step 9: oracle on dev (100 questions)

In [ ]:
!python scripts/run_llm.py --system oracle --split dev --out {RESULTS_DIR} --cache-dir {LLM_CACHE} --hf-cache-dir {HF_CACHE}

## Step 10: summary

In [ ]:
import json

for system in ("closed_book", "oracle"):
    with open(f"{RESULTS_DIR}/{system}/dev_metrics.json") as f:
        m = json.load(f)
    print(system, {k: m[k] for k in ("accuracy", "macro_f1", "n_parse_failures")})
    print("   ", m["llm"])

## Step 11: download the results

In [ ]:
!cd {DRIVE_ROOT}/results && zip -qr /content/llm_results.zip llm
from google.colab import files

files.download("/content/llm_results.zip")

## What to send back

Unzip `llm_results.zip` into the local repo so the files land in `results/llm/closed_book/` and
`results/llm/oracle/`. The response cache stays on Drive (it is reused in Phase 3 and 4 and is
never committed).